In [2]:
import pandas as pd
import duckdb
import json
import os

# -----------------------------
# 경로
# -----------------------------
train_path = "../data/fs_sample_data/fs_sample_train.parquet"
test_path  = "../data/fs_sample_data/fs_sample_validation.parquet"

REMOVE_LISTS = [
    r'..\config\remove_features_v1.json',
    r'..\config\remove_features_v2.json',
]

TRAIN_OUT = "../data/fs_sample_data/fs_train.parquet"
TEST_OUT  = "../data/fs_sample_data/fs_validation.parquet"


# -----------------------------
# 제외 리스트 로드
# -----------------------------
def get_exclusion_list(paths):
    exclude_set = set()
    for path in paths:
        if not os.path.exists(path):
            continue
        with open(path, 'r', encoding='utf-8') as f:
            cfg = json.load(f)
            for key in cfg:
                if isinstance(cfg[key], list):
                    exclude_set.update(cfg[key])
    return list(exclude_set)


# -----------------------------
# 데이터 로드
# -----------------------------
train_df = duckdb.query(f"SELECT * FROM '{train_path}'").df()
test_df  = duckdb.query(f"SELECT * FROM '{test_path}'").df()


# -----------------------------
# 컬럼 제거
# -----------------------------
base_drop = ['date', 'serial_number']   # 필요 없으면 제거
json_drop = get_exclusion_list(REMOVE_LISTS)

drop_cols = list(set(base_drop + json_drop))

train_df = train_df.drop(columns=drop_cols, errors='ignore')
test_df  = test_df.drop(columns=drop_cols, errors='ignore')


# -----------------------------
# 저장
# -----------------------------
train_df.to_parquet(TRAIN_OUT, index=False)
test_df.to_parquet(TEST_OUT, index=False)

print("완료")

완료
